In [1]:
from pyspark.sql import SparkSession

In [2]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta

import logging

from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_1762/3320559344.py:10: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
spark = SparkSession.builder \
    .appName("mob1") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "3")\
    .config('spark.executor.memory', '7g')\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 20:18:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
def build_gr_mobile_datamart(spark, logical_date, logger):
    from pyspark.sql import functions as F

    business_month_end = logical_date + relativedelta(months=1, days=-1)

    cp_mobile_daily_history_df = spark.table('gr_dm_sec.cp_mobile_daily_history')
    
    daily_features = (
        cp_mobile_daily_history_df
        .limit(1_000_000)
        .withColumn('tp_change_days',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('tp_change_dt')))
        .withColumn('max_subs_fee_day_31d_days_ago',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('max_subs_fee_dt_day_31d')))
        .withColumn('ep_tp_change_days',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('ep_tp_change_dt')))
        .withColumn('device_first_imei_msisdn_days',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('device_first_imei_msisdn_dt')))
        .withColumn('device_first_imei_msisdn_days',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('device_first_imei_msisdn_dt')))
        .withColumn('last_change_smartphone_days',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('last_change_smartphone_dt')))
        .withColumn('start_use_cur_smartp_mod_days',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('start_use_cur_smartp_mod_dt')))
        .withColumn('mymts_max_activity_master_days',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('mymts_max_activity_master_dt')))
        .withColumn('mymts_max_activity_slave_days',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('mymts_max_activity_slave_dt')))
        .withColumn('mnp_portation_in_mts_days',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('mnp_portation_in_mts_dt')))
        .withColumn('last_active_roam_day_90d_days_ago',
                    F.datediff(F.to_date(F.lit(business_month_end)), F.to_date('last_active_roam_dt_day_90d')))

        .limit(1_000_000)
    )

    res_df = (
        daily_features
        .join(spark.table("gr_dm.cp_mobile_monthly_history").drop('sappn', 'sappn_str', 'm_reg_cd', 'table_business_month'), ['app_n', 'regid'], 'left')
    )

    return res_df

In [5]:
logical_date = datetime.now().date()
log = logging.getLogger(__name__)

In [6]:
build_gr_mobile_datamart(spark, logical_date, log)

DataFrame[msisdn: int, app_n: int, regid: int, tp_change_dt: date, max_subs_fee_dt_day_31d: date, ep_tp_change_dt: date, device_first_imei_msisdn_dt: date, last_change_smartphone_dt: date, start_use_cur_smartp_mod_dt: date, mymts_max_activity_master_dt: date, mymts_max_activity_slave_dt: date, mnp_portation_in_mts_dt: date, last_active_roam_dt_day_90d: date, bill_group_name: int, lifetime: int, m_categ_name: int, m_reg_cd: int, m_reg_name: int, sale_channel_cd: int, sale_channel_name: int, fl_client_name_confirmed: int, client_type_cd: int, sex: int, age: int, citizenship_type: int, foreign_citizenship_cd: int, region_name: int, msisdn_type_cd: int, msisdn_cluster: int, trust_category: int, credit_limit_value: int, cnt_app_on_acc: int, service_provider_cd: int, fl_push_off: int, fl_unlim_internet: int, max_subs_fee_day_31d: decimal(10,2), host_type: int, activity_on_bts_dur_day_30d: int, cnt_give_out_prizes_day_7d: int, cnt_give_out_prizes_day_30d: int, cnt_give_out_prizes_day_90d: int

In [7]:
build_gr_mobile_datamart(spark, logical_date, log).count()

26/03/16 20:06:15 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/03/16 20:06:30 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
                                                                                

500000

In [9]:
final_df = build_gr_mobile_datamart(spark, logical_date, log)
(
    final_df
    .repartition(4)
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("spfix_dm.mob_features_dm__1__base")
)

26/03/16 20:06:47 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/16 20:07:18 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/03/16 20:11:50 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/03/16 20:11:50 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:978)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(

In [8]:
spark.stop()